### General Imports:

In [3]:
import os
import gym
from stable_baselines3.ppo import PPO
from stable_baselines3.ppo.policies import MlpPolicy as MLP_PPO
from netsim.netSimPy import *
from netsim.gym_basic.envs import RMSA_ENV
from stable_baselines3.common.monitor import Monitor
import numpy as np
import tensorflow as tf
import logging
from IPython.display import clear_output
logging.getLogger('tensorflow').setLevel(logging.FATAL)
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import sync_envs_normalization
# from netsim.allocators import sap_ff, ldpb, sap_ff2
from stable_baselines3.common.evaluation import evaluate_policy
from netsim.allocators import sap_ff
tf.__version__


'2.12.0'

In [4]:
def warming_lr(initial_lr=1.5e-4, max_warm_lr = 3e-4, final_lr=2e-4, warmup_progress = 0.85):
    def learning_rate_fn(progress):
        if progress >= warmup_progress:
            m = (initial_lr-max_warm_lr)/(1-warmup_progress)
            n = initial_lr - m
            return round(m*progress+n,6)
        else:
            m = (max_warm_lr-final_lr)/ warmup_progress
            return round(m*progress + final_lr,6)
    
    return learning_rate_fn

In [5]:
from typing import Union, Optional

class MyBestCallback(EvalCallback):
    def __init__(
        self,
        eval_env,
        callback_on_new_best = None,
        callback_after_eval = None,
        n_eval_episodes: int = 5,
        eval_freq: int = 10000,
        log_path: Optional[str] = None,
        best_model_save_path: Optional[str] = None,
        deterministic: bool = True,
        render: bool = False,
        verbose: int = 1,
        warn: bool = True,
    ):
        super().__init__(
            eval_env,
            callback_on_new_best,
            callback_after_eval,
            n_eval_episodes,
            eval_freq,
            log_path,
            best_model_save_path,
            deterministic,
            render,
            verbose,
            warn
        )
        self.eval_env = eval_env
        self.best_reward_at = 0

    def _on_step(self) -> bool:
        continue_training = True

        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            # Sync training and eval env if there is VecNormalize
            if self.model.get_vec_normalize_env() is not None:
                try:
                    sync_envs_normalization(self.training_env, self.eval_env)
                except AttributeError as e:
                    raise AssertionError(
                        "Training and eval env are not wrapped the same way, "
                        "see https://stable-baselines3.readthedocs.io/en/master/guide/callbacks.html#evalcallback "
                        "and warning above."
                    ) from e

            # Reset success rate buffer
            self._is_success_buffer = []
            # self.eval_env.reset(hard_reset=True)
            episode_rewards, episode_lengths = evaluate_policy(
                self.model,
                self.eval_env,
                n_eval_episodes=self.n_eval_episodes,
                render=self.render,
                deterministic=self.deterministic,
                return_episode_rewards=True,
                warn=self.warn,
                callback=self._log_success_callback,
            )

            if self.log_path is not None:
                self.evaluations_timesteps.append(self.num_timesteps)
                self.evaluations_results.append(episode_rewards)
                self.evaluations_length.append(episode_lengths)

                kwargs = {}
                # Save success log if present
                if len(self._is_success_buffer) > 0:
                    self.evaluations_successes.append(self._is_success_buffer)
                    kwargs = dict(successes=self.evaluations_successes)

                np.savez(
                    self.log_path,
                    timesteps=self.evaluations_timesteps,
                    results=self.evaluations_results,
                    ep_lengths=self.evaluations_length,
                    **kwargs,
                )

            mean_reward, std_reward = np.mean(episode_rewards), np.std(episode_rewards)
            mean_ep_length, std_ep_length = np.mean(episode_lengths), np.std(episode_lengths)
            self.last_mean_reward = mean_reward

            if self.verbose >= 1:
                print(f"Eval num_timesteps={self.num_timesteps}, " f"episode_reward={mean_reward:.2f} +/- {std_reward:.2f}")
            # Add to current Logger
            self.logger.record("eval/mean_reward", float(mean_reward))
            self.logger.record("eval/mean_ep_length", mean_ep_length)

            if len(self._is_success_buffer) > 0:
                success_rate = np.mean(self._is_success_buffer)
                if self.verbose >= 1:
                    print(f"Success rate: {100 * success_rate:.2f}%")
                self.logger.record("eval/success_rate", success_rate)

            # Dump log so the evaluation results are printed with the correct timestep
            self.logger.record("time/total_timesteps", self.num_timesteps, exclude="tensorboard")
            self.logger.dump(self.num_timesteps)

            if mean_reward >= self.best_mean_reward:
                self.best_reward_at = self.num_timesteps
                if self.best_model_save_path is not None:
                    self.model.save(os.path.join(self.best_model_save_path, "best_model"))
                self.best_mean_reward = mean_reward
                # Trigger callback on new best model, if needed
                if self.callback_on_new_best is not None:
                    continue_training = self.callback_on_new_best.on_step()
            print(
                f"Best mean reward: {self.best_mean_reward:.2f} at timestep No. {self.best_reward_at}"
            )
            clear_output(wait=True)
            # Trigger callback after every evaluation, if needed
            if self.callback is not None:
                continue_training = continue_training and self._on_event()

        return continue_training

### Training:

In [12]:
from netsim.allocators import sap_ff
__file__ = 'RMSA.ipynb'
absolutepath = os.path.abspath(__file__)
file_name = os.path.basename(os.path.abspath(__file__)).split('.')[0]
fileDirectory = os.path.dirname(os.path.dirname(os.getcwd())) + "/notebooks/networks"
log_dir=f'./tmp/{file_name}/'
os.makedirs(log_dir, exist_ok=True)
tensorboard_log = f"./tb/{file_name}/nsfnet/"

M_LAMBDA = 10000
network = Network(
    networkFileName =  "../../networks/nsfnet/network_c.json",
    pathsFileName= "../../networks/nsfnet/routes.json",
    bitrateFilename= "../../networks/nsfnet/bitrates_c_bands.json",
)
generator = EventsGenerator(mLambda=M_LAMBDA)


N_EVALUATIONS_EPISODES = 5
EPISODE_LENGTH = 100
EVAL_FREQ = N_EVALUATIONS_EPISODES*EPISODE_LENGTH

sim_args = dict(
    network = network,
    eventsGenerator=generator,
    # allocator=sap_ff(3)
)
simulator = Simulator(**sim_args)

env_args = dict(
    simulator = simulator,
    episode_length = EPISODE_LENGTH,
    allocator=sap_ff(3)
)
env = Monitor(gym.make("RMSA_ENV-v0", **env_args), log_dir)



# # MODEL HYPERPARAMETERS:
mArgs = dict(
    # gamma=0.95,
    # learning_rate=0.0001,
    # n_steps= 4096,
    # batch_size = 32,
    # n_epochs=20,
)
model = PPO(MLP_PPO, env, verbose=0, seed=3, tensorboard_log=tensorboard_log, **mArgs)

callback = MyBestCallback(
    env,
    n_eval_episodes=N_EVALUATIONS_EPISODES, 
    eval_freq=EVAL_FREQ,
    log_path=log_dir, 
    best_model_save_path=log_dir,
    verbose=1)
train_model = model.learn(total_timesteps=2000000, callback=callback)
# simulator.run(20000)
#TODO: tiene un error el algoritmo de allocator del enviroment



KeyboardInterrupt: 

In [ ]:
# train_model = PPO.load(f"./tmp/{file_name}/best_model.zip")
# env.setLambda(100000)

In [ ]:

# mean_reward, std_reward = evaluate_policy(train_model, env, n_eval_episodes=100, deterministic=True)
# print(mean_reward)

In [ ]:
# env.setAllocatorFunc(sap_ff2(3))
# mean_reward, std_reward = evaluate_policy(train_model, env, n_eval_episodes=100, deterministic=True)
# print(mean_reward)